# Heart Disease: Model Selection

In this notebook we compare three models fairly, using cross-validation on the training set only. The test set stays untouched until we have chosen our final model, so that its score is an honest estimate of performance on unseen patients.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("../data/heart_raw.csv")
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
categorical = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])
print(X_train.shape, X_test.shape)

(242, 13) (61, 13)


## 1. Comparing models with cross-validation

With only 303 patients, a single train/test score is noisy. Instead we use repeated stratified 5-fold cross-validation on the training set: the data is split into 5 folds several times, and each model is scored on held-out folds. We report the mean and spread of accuracy, recall, F1 and ROC-AUC.

In [2]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Random forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                             eval_metric="logloss", random_state=42),
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scoring = ["accuracy", "recall", "f1", "roc_auc"]

rows = []
for name, clf in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("clf", clf)])
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({
        "model": name,
        **{m: f"{res[f'test_{m}'].mean():.3f} ± {res[f'test_{m}'].std():.3f}" for m in scoring},
    })

pd.DataFrame(rows).set_index("model")

,accuracy,recall,f1,roc_auc
model,,,,
Logistic regression,0.825 ± 0.049,0.765 ± 0.082,0.800 ± 0.057,0.892 ± 0.043
Random forest,0.799 ± 0.058,0.759 ± 0.093,0.774 ± 0.069,0.887 ± 0.043
XGBoost,0.778 ± 0.049,0.759 ± 0.101,0.756 ± 0.062,0.861 ± 0.045
